# Rapport final : Python pour la data-science 2A

Manon Bourguignat, Leila Héry, Franck Tran  
Chargé de TD : Lino Galiana

On pensera à lire avant le fichier README.md du repo qui contient notamment les informations sur les dépendances et les données.

Ce document constitue le rapport final du projet.  Tout se lit et sexecute de manière linéaire. 

De nombreuses fonctions utilisées ici sont regroupées dans des scripts python, dans le dossier "scripts".

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.preparation_data import recuperer_donnees, dico_doi, select_colonne, suppr_colonne, renommer_colonne



## I. Contexte

### 1. Présentation du sujet

ecrire ici contexte + problématisation du sujet (bien regarder quels variables on va utilsier pour faire la problématique à laquelel "on va répondre", faire une intro similaire au pdf de statapp, que je n'ai pas pensé à telecharger...)

### 2. Présentation de la base de donnée utilisée

L'European Social Service (ESS) est une enquête menée régulièrement dans plusieurs pays européens pour étudier les opinions, les valeurs et les comportements des populations. Chaque round comprend des variables communes à toutes les enquêtes, ainsi que des variables spécifiques aux thèmes étudiés.

Dans ce projet, on utilisera les données de l’ESS Round 10 (noté égament ESS10), édition 3.3, collectées en 2020 dont les thèmes sont la démocratie et les contacts sociaux numériques.

Tous les détails (références, licenses, accès) concernant les données se situent dans le fichier README.md.

## II. Préparation des données

Cette partie consiste en la récupération, l'exploration et le nettoyage des données.

### 1. Récupération des données

Le choix a été de récupèrer les données de l'enquête de 2020 avec l'API dédiée pour deux raisons :
- Faciliter la reproductibilité : l'API est plus pratique que de chercher à récupérer les données sur le site
- L'infrastructure de recherche qui gère l'ESS recommande dans ses conditions d'utilisation (https://europeansocialsurvey.org/contact/disclaimer) de ne pas mettre les jeux de données de l’ESS à disposition sur des sites web externes.

Lorsqu’on exécute la fonction ci-dessous et définie dans preparation_data.py, **il faut renseigner un "ESS User ID"**, disponible sur la page de l’API de l’ESS (https://ess.sikt.no/en/api) une fois connecté. On pourra également utiliser l'ESS User ID de Franck TRAN qu'il vous aura transmis par une autre voie. Cet identifiant ne sert pas à s’authentifier, mais à suivre les statistiques d’utilisation des données. 

In [34]:
df_ess_2020 = recuperer_donnees(2020)
print(df_ess_2020.head(3))

         name  essround edition    proddate   idno cntry   dweight   pspwght  \
0  ESS10e03_3        10     3.3  02.12.2025  10038    BE  0.882220  0.972276   
1  ESS10e03_3        10     3.3  02.12.2025  10053    BE  1.047642  0.888635   
2  ESS10e03_3        10     3.3  02.12.2025  10055    BE  1.087741  0.722811   

    pweight  anweight  ...               vinwe               inwde  \
0  0.718075  0.698167  ... 2022-09-01 17:47:00 2022-09-01 17:47:00   
1  0.718075  0.638107  ... 2022-04-08 11:07:00 2022-04-08 11:10:00   
2  0.718075  0.519033  ... 2022-05-20 11:08:00 2022-05-20 11:10:00   

                jinws               jinwe  inwtm  mode  domain      prob  \
0 2022-09-01 17:47:00 2022-09-01 17:47:00   36.0   1.0       1  0.000397   
1 2022-04-08 11:07:00 2022-04-08 11:10:00   54.0   2.0       2  0.000334   
2 2022-05-20 11:08:00 2022-05-20 11:10:00   77.0   1.0       2  0.000322   

   stratum   psu  
0      188  2596  
1      194  2206  
2      198  2114  

[3 rows x 621 co

On conserve l’ensemble des variables susceptibles de contribuer à la réponse à la problématique et on les renomme pour les rendre plus explicites :

In [39]:
colonnes_a_garder = [
    # --- 1) identifiants et pondérations méthodologiques ---
    "idno",       # Identifiant unique de l'individu
    "cntry",      # Pays de résidence (essentiel pour les comparaisons européennes)
    "anweight",   # Poids d'analyse (corrige les biais d'échantillonnage nationaux)
    "dweight",    # Poids de sondage (design weight)
    "pspwght",    # Poids post-stratification

    # --- 2) Ccoyances aux théories du complot & attitudes face à la science ---
    "secgrdec",   # Groupes secrets qui influencent/dirigent les décisions politiques
    "scidecpb",   # Scientifiques et figures politiques qui s'entendent en secret
    "gvconc19",   # Dissimulation par le gouvernement d'informations clés sur le Covid-19
    "gvimpc19",   # Intention réelle derrière les mesures sanitaires (contrôle vs santé)
    "gvbalc19",   # Équilibre des décisions sanitaires perçu (santé vs intérêts)
    "trstsci",    # Confiance dans la science et les scientifiques
    "getnvc19",   # Raisons ou volonté de refus de la vaccination Covid-19
    "icvacc19",   # Statut vaccinal Covid-19 (a reçu le vaccin oui/non)

    # --- 3. positionnement politique & confiance institutionnelle ---
    "lrscale",    # Positionnement politique sur l'axe gauche-droite (échelle 0-10)
    "polintr",    # Degré d'intérêt pour la politique
    "vote",       # Participation électorale aux dernières élections
    "clsprty",    # Sentiment de proximité avec un parti politique en particulier
    "prtdgcl",    # Degré d'attachement à ce parti
    "trstprl",    # Confiance dans le parlement national
    "trstplt",    # Confiance dans les hommes et femmes politiques
    "trstprt",    # Confiance dans les partis politiques
    "trstlgl",    # Confiance dans la justice / le système judiciaire
    "trstplc",    # Confiance dans la police
    "cptppola",   # Sentiment d'efficacité politique (capacité des citoyens à peser sur le système)
    "psppipla",   # Sentiment que le système politique prend en compte les citoyens

    # --- 4. éducation & capital socio-économique ---
    "eisced",     # Plus haut diplôme obtenu (standard européen ES-ISCED)
    "eduyrs",     # Nombre d'années complètes d'études
    "hinctnta",   # Revenu net total du ménage (en déciles nationaux)
    "hincfel",    # Perception du niveau de revenu (vit confortablement / difficultés)
    "mainact",    # Activité principale (emploi, études, chômage, retraite...)
    "uempla",     # Situation de recherche d'emploi / chômage
    "domicil",    # Degré d'urbanisation de la zone de vie (métropole, banlieue, rural...)

    # --- 5. religion, croyances & spiritualité ---
    "rlgblg",     # Sentiment d'appartenance à une religion particulière (oui/non)
    "rlgdgr",     # Intensité du sentiment religieux (échelle 0-10)
    "rlgatnd",    # Fréquence de fréquentation des lieux de culte
    "pray",       # Fréquence de la prière en dehors des offices religieux

    # --- 6. données démographiques fondamentales ---
    "gndr",       # Genre (1 = Homme, 2 = Femme)
    "agea",       # Âge exact du répondant
    "yrbrn",      # Année de naissance
]

# Noms explicites en français correspondants exactement à colonnes_a_garder
nouveaux_noms = [
    # 1) identifiants & poids
    "id", "pays", "poids_analyse", "poids_design", "poids_poststrat",
    # 2) Ccoyances complotistes & science
    "complot_groupes_secrets", "complot_science_politique", "dissimulation_covid", 
    "mesures_sanitaires_controle", "mesures_sanitaires_equilibre", "confiance_science", 
    "refus_vaccin", "statut_vaccine",
    # 3) politique & institutions
    "axe_gauche_droite", "interet_politique", "a_vote", "proche_parti", "degre_proximite_parti",
    "confiance_parlement", "confiance_politiques", "confiance_partis", "confiance_justice", 
    "confiance_police", "efficacite_politique_citoyen", "prise_en_compte_citoyen",
    # 4) éducation & socio-éco
    "niveau_etudes_isced", "annees_etudes", "decile_revenu", "sentiment_revenu", 
    "activite_principale", "chomage", "milieu_residence",
    # 5) religion & spiritualité
    "appartenance_religieuse", "degre_religiosite", "frequence_culte", "frequence_priere",
    # 6) démographie
    "genre", "age", "annee_naissance"
]

df_ess_select = select_colonne(df_ess_2020, colonnes_a_garder)
df_ess_rename = renommer_colonne(df_ess_select, colonnes_a_garder, nouveaux_noms)
print(df_ess_rename.head(3))

      id pays  poids_analyse  poids_design  poids_poststrat  \
0  10038   BE       0.698167      0.882220         0.972276   
1  10053   BE       0.638107      1.047642         0.888635   
2  10055   BE       0.519033      1.087741         0.722811   

   complot_groupes_secrets  complot_science_politique  dissimulation_covid  \
0                      4.0                        3.0                  4.0   
1                      3.0                        3.0                  3.0   
2                      4.0                        5.0                  5.0   

   mesures_sanitaires_controle  mesures_sanitaires_equilibre  ...  \
0                          7.0                           2.0  ...   
1                          4.0                           6.0  ...   
2                          7.0                           5.0  ...   

   activite_principale  chomage  milieu_residence  appartenance_religieuse  \
0                  NaN        0               4.0                      1.0   
1

### 2. Valeurs manquantes

Regardons la part des valeurs manquantes pour chaque variable.

In [51]:
df_val_manquant = pd.DataFrame({
    "Nbr valeurs manquantes": df_ess_rename.isnull().sum(),
    "% valeurs manquantes": df_ess_rename.isnull().mean() * 100
})

df_val_manquant = df_val_manquant.sort_values(by="% valeurs manquantes", ascending=False)

print(df_val_manquant.head(10))

                           Nbr valeurs manquantes  % valeurs manquantes
refus_vaccin                                37073             98.569567
activite_principale                         33940             90.239558
degre_proximite_parti                       23289             61.920715
statut_vaccine                              20679             54.981255
decile_revenu                                8239             21.905825
confiance_science                            7866             20.914094
dissimulation_covid                          5762             15.319986
complot_groupes_secrets                      5632             14.974343
axe_gauche_droite                            5297             14.083646
complot_science_politique                    5293             14.073011


(Eventuelement mettre une courte interpretation de pourquoi les 4 premières variables ont autant de valeurs manquantes) 

On n'utilisera ainsi pas les variables refus_vaccin, activite_principale, degre_proximite_parti, statut_vaccine qui ont beaucoup trop de valeurs manquantes :

In [52]:
df_ess_suppr = suppr_colonne(df_ess_rename, ["refus_vaccin", "activite_principale",
                        "degre_proximite_parti", "statut_vaccine"])

print(df_ess_suppr.shape)

(37611, 35)


De nombreuses variables qu'on ne peut pas retirer dans le cadre de ce projet comme complot_groupes_secrets, confiance_science, dissimulation_covid, complots_groupe_secrets ont entre 10% et 20% de valeurs manquantes, ce qui est relativement élevé. On remarque cependant que cela est dû au fait que ces variables ont souvent 100% de valeurs manquantes dans certains pays :

(Si qqn se sent d'attak pour voir la documentation pour voir sil y a une explication de pk la France et monténégro ya pas ces varaibles et ecrire ici la raison)

In [56]:
df_par_pays = df_ess_suppr.groupby("pays").agg(lambda col: col.isnull().mean() * 100)

df_par_pays = df_par_pays[["decile_revenu", "confiance_science", "dissimulation_covid",
    "complot_groupes_secrets", "axe_gauche_droite", "complot_science_politique"]]

df_par_pays = df_par_pays.sort_values(by="confiance_science", ascending=False)

print(df_par_pays)


      decile_revenu  confiance_science  dissimulation_covid  \
pays                                                          
EE         2.594034         100.000000             0.907912   
CZ        30.452342         100.000000             6.744750   
ME        10.876369         100.000000           100.000000   
FR        10.723318         100.000000           100.000000   
PT        34.276387           5.277476            23.014146   
MK        23.582925           3.778866             7.837649   
IE        32.994350           3.446328            11.073446   
CH        17.662508           3.348654             7.025607   
LT        16.937914           2.772755             6.992164   
NL        10.340136           2.517007             2.585034   
HU        26.933478           2.217415            16.711736   
HR        30.527638           2.198492             6.469849   
SK        29.971791           2.045134            11.424542   
BG        16.629875           1.545254             6.14

On remarque ainsi que les valeurs manquent principalement à cause de la France/monténégro (reformuler cette phrase). Il est donc probablement plus judicieux de retirer la France et le Monténégro de notre analyse, quitte à avoir moins de données.

In [64]:
df_ess_suppr_pays = df_ess_suppr[~df_ess_suppr["pays"].isin(["FR", "ME"])]

print(df_ess_suppr_pays.shape)

(34356, 35)


Regardons à nouveau si la part des valeurs manquantes a diminué :

In [65]:
df_val_manquant_2 = pd.DataFrame({
    "Nbr valeurs manquantes": df_ess_suppr_pays.isnull().sum(),
    "% valeurs manquantes": df_ess_suppr_pays.isnull().mean() * 100
})

df_val_manquant_2 = df_val_manquant_2.sort_values(by="% valeurs manquantes", ascending=False)

print(df_val_manquant_2.head(10))

                              Nbr valeurs manquantes  % valeurs manquantes
decile_revenu                                   7888             22.959599
axe_gauche_droite                               4662             13.569682
confiance_science                               4611             13.421236
dissimulation_covid                             2507              7.297124
complot_groupes_secrets                         2377              6.918733
complot_science_politique                       2038              5.932006
mesures_sanitaires_equilibre                    1839              5.352777
proche_parti                                     825              2.401327
efficacite_politique_citoyen                     794              2.311096
mesures_sanitaires_controle                      673              1.958901


Faire un commentaire sur les résultats (est ce que la part des valeurs manquantes a bien baissé ? )

Remplacer valeurs manquante par mean ou median ou les drop ? Regarder les types de variables. Normaliser les variables ? Affiche la taille, decrire dataframe final 

PS : g pas fini

### 3. Création de variables continues/agrégées

Créer de nouvelles variables qui réunit les autres

In [ ]:
# ici

## III. Visualisation 

Cette partie consiste en l'analyse et la visualsation...............

### 1. Statistiques descriptives

In [ ]:
# faire dataframe ou graphiques comme dans la 6 du fichier stats desc qui monte les stats comme moyenne, etc. 
# des différentes variables

Faire des "commentaires dessus"

### 2. Analyse descriptive

Graphiques basique (histogramme, boxplot qui permet situer)

Graphiques détaillant relation entre les variables (genre confiance_science en fonction du decile revenu etc..)

## IV. Conclusion

Ceci est une conclusion